# 🚦 Traffic Demand Prediction
**Flipkart AI/ML Hackathon**  
**Evaluation Metric:** R² Score (higher is better)  
**Task:** Predict continuous `demand` variable  

---
### Strategy
1. EDA & Data Inspection
2. Feature Engineering (time, geohash, categorical, target encoding)
3. 5-Fold Cross-Validated LightGBM
4. XGBoost ensemble
5. Weighted blend + submission

In [ ]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
print('Libraries loaded ✅')

In [ ]:
# ─── 2. LOAD DATA ─────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'Sample sub  : {sample_sub.shape}')
train.head()

In [ ]:
# ─── 3. EDA ───────────────────────────────────────────────────────────────────
print('=== DTYPES ===')
print(train.dtypes)
print()
print('=== MISSING VALUES (train) ===')
print(train.isnull().sum())
print()
print('=== DEMAND STATS ===')
print(train['demand'].describe())

In [ ]:
# Categorical unique values
for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
    print(f'{col}: {train[col].unique()}')

In [ ]:
# Demand distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train['demand'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('demand')

# Hourly demand pattern
train_tmp = train.copy()
train_tmp['hour'] = train_tmp['timestamp'].apply(lambda x: int(x.split(':')[0]))
hourly = train_tmp.groupby('hour')['demand'].mean()
axes[1].plot(hourly.index, hourly.values, marker='o', color='coral')
axes[1].set_title('Average Demand by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Demand')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Demand by categorical features
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']):
    train.groupby(col)['demand'].mean().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Demand by {col}')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ─── 4. PRECOMPUTE TARGET-ENCODING STATS (on full train) ──────────────────────

# Geohash-level demand stats
geo_stats = train.groupby('geohash')['demand'].agg(
    geo_mean='mean',
    geo_median='median',
    geo_std='std',
    geo_q25=lambda x: x.quantile(0.25),
    geo_q75=lambda x: x.quantile(0.75),
    geo_max='max'
).reset_index()
geo_stats['geo_std'] = geo_stats['geo_std'].fillna(0)

# Geohash + Hour mean demand
_tmp = train.copy()
_tmp['hour'] = _tmp['timestamp'].apply(lambda x: int(x.split(':')[0]))
geo_hour_stats = _tmp.groupby(['geohash', 'hour'])['demand'].mean().reset_index()
geo_hour_stats.columns = ['geohash', 'hour', 'geo_hour_mean']

# Geohash + Day mean demand
geo_day_stats = train.groupby(['geohash', 'day'])['demand'].mean().reset_index()
geo_day_stats.columns = ['geohash', 'day', 'geo_day_mean']

# Global median (for unseen geohashes in test)
GLOBAL_DEMAND_MEDIAN = train['demand'].median()
GLOBAL_TEMP_MEDIAN   = train['Temperature'].median()

print('Stats computed ✅')
print(f'Unique geohashes in train: {train["geohash"].nunique()}')
print(f'Unique geohashes in test : {test["geohash"].nunique()}')

In [ ]:
# ─── 5. FEATURE ENGINEERING FUNCTION ─────────────────────────────────────────

def engineer_features(df, geo_stats, geo_hour_stats, geo_day_stats, global_temp_median):
    df = df.copy()
    
    # --- Time Features ---
    df['hour']        = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute']      = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_of_day'] = df['hour'] * 60 + df['minute']   # 0-1425
    
    # Cyclical encoding of time (captures midnight wrap-around)
    df['time_sin'] = np.sin(2 * np.pi * df['time_of_day'] / (24 * 60))
    df['time_cos'] = np.cos(2 * np.pi * df['time_of_day'] / (24 * 60))
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Peak / off-peak flags
    df['is_peak_morning'] = df['hour'].isin([7, 8, 9]).astype(int)
    df['is_peak_midday']  = df['hour'].isin([10, 11, 12, 13, 14]).astype(int)
    df['is_peak_evening'] = df['hour'].isin([17, 18, 19]).astype(int)
    df['is_night']        = df['hour'].isin([0, 1, 2, 3, 4]).astype(int)
    df['is_peak']         = ((df['is_peak_morning'] + df['is_peak_midday'] + df['is_peak_evening']) > 0).astype(int)
    df['quarter_of_day']  = df['hour'] // 6   # 0=midnight, 1=morning, 2=afternoon, 3=evening
    
    # --- Categorical Encoding ---
    df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_enc']     = (df['Landmarks'] == 'Yes').astype(int)
    
    road_map    = {'Highway': 2, 'Street': 1, 'Residential': 0}
    weather_map = {'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3}
    df['RoadType_enc'] = df['RoadType'].map(road_map).fillna(-1)
    df['Weather_enc']  = df['Weather'].map(weather_map).fillna(-1)
    
    # --- Temperature ---
    df['Temperature'] = df['Temperature'].fillna(global_temp_median)
    df['temp_sq']     = df['Temperature'] ** 2
    df['temp_abs']    = df['Temperature'].abs()
    
    # --- Geohash Features ---
    df['geo_prefix3'] = df['geohash'].str[:3]
    df['geo_prefix4'] = df['geohash'].str[:4]
    
    # Geohash target encoding
    df = df.merge(geo_stats, on='geohash', how='left')
    for col in ['geo_mean', 'geo_median', 'geo_std', 'geo_q25', 'geo_q75', 'geo_max']:
        df[col] = df[col].fillna(GLOBAL_DEMAND_MEDIAN)
    
    # Geohash + Hour mean demand (most powerful feature)
    df = df.merge(geo_hour_stats, on=['geohash', 'hour'], how='left')
    df['geo_hour_mean'] = df['geo_hour_mean'].fillna(df['geo_mean'])
    
    # Geohash + Day mean demand
    df = df.merge(geo_day_stats, on=['geohash', 'day'], how='left')
    df['geo_day_mean'] = df['geo_day_mean'].fillna(df['geo_mean'])
    
    # Interaction: geo_mean * peak flag
    df['geo_mean_x_peak']  = df['geo_mean'] * df['is_peak']
    df['geo_mean_x_lanes'] = df['geo_mean'] * df['NumberofLanes']
    
    return df

print('Feature engineering function defined ✅')

In [ ]:
# ─── 6. APPLY FEATURES ────────────────────────────────────────────────────────
train_fe = engineer_features(train, geo_stats, geo_hour_stats, geo_day_stats, GLOBAL_TEMP_MEDIAN)
test_fe  = engineer_features(test,  geo_stats, geo_hour_stats, geo_day_stats, GLOBAL_TEMP_MEDIAN)

FEATURES = [
    # Time
    'hour', 'minute', 'time_of_day',
    'time_sin', 'time_cos', 'hour_sin', 'hour_cos',
    'is_peak_morning', 'is_peak_midday', 'is_peak_evening',
    'is_night', 'is_peak', 'quarter_of_day',
    # Road / Vehicle
    'LargeVehicles_enc', 'Landmarks_enc', 'RoadType_enc', 'NumberofLanes',
    # Weather
    'Weather_enc', 'Temperature', 'temp_sq', 'temp_abs',
    # Time
    'day',
    # Geohash target encoding
    'geo_mean', 'geo_median', 'geo_std', 'geo_q25', 'geo_q75', 'geo_max',
    'geo_hour_mean', 'geo_day_mean',
    # Interactions
    'geo_mean_x_peak', 'geo_mean_x_lanes',
]

X      = train_fe[FEATURES].values
y      = train_fe['demand'].values
X_test = test_fe[FEATURES].values

print(f'Feature matrix: X={X.shape}, X_test={X_test.shape}')
print(f'Target range: [{y.min():.6f}, {y.max():.6f}]')

In [ ]:
# ─── 7. LIGHTGBM — 5-FOLD CV ──────────────────────────────────────────────────
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

lgb_params = dict(
    n_estimators     = 3000,
    learning_rate    = 0.02,
    num_leaves       = 255,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    random_state     = SEED,
    n_jobs           = -1,
    verbose          = -1,
)

fold_scores_lgb = []
print('Training LightGBM...')

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)]
    )
    
    preds_val          = np.clip(model.predict(X_val), 0, 1)
    oof_lgb[val_idx]   = preds_val
    test_lgb          += np.clip(model.predict(X_test), 0, 1) / N_FOLDS
    
    score = r2_score(y_val, preds_val)
    fold_scores_lgb.append(score)
    print(f'  Fold {fold+1} R²: {score:.6f}  |  best iter: {model.best_iteration_}')

lgb_oof_score = r2_score(y, np.clip(oof_lgb, 0, 1))
print(f'\n✅ LightGBM OOF R²: {lgb_oof_score:.6f}  =>  Competition Score: {max(0,100*lgb_oof_score):.4f}')

In [ ]:
# ─── 8. XGBOOST — 5-FOLD CV ───────────────────────────────────────────────────
oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

xgb_params = dict(
    n_estimators     = 3000,
    learning_rate    = 0.02,
    max_depth        = 8,
    min_child_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    random_state     = SEED,
    n_jobs           = -1,
    tree_method      = 'hist',
    verbosity        = 0,
)

fold_scores_xgb = []
print('Training XGBoost...')

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
        early_stopping_rounds=150
    )
    
    preds_val          = np.clip(model.predict(X_val), 0, 1)
    oof_xgb[val_idx]   = preds_val
    test_xgb          += np.clip(model.predict(X_test), 0, 1) / N_FOLDS
    
    score = r2_score(y_val, preds_val)
    fold_scores_xgb.append(score)
    print(f'  Fold {fold+1} R²: {score:.6f}  |  best iter: {model.best_iteration}')

xgb_oof_score = r2_score(y, np.clip(oof_xgb, 0, 1))
print(f'\n✅ XGBoost OOF R²: {xgb_oof_score:.6f}  =>  Competition Score: {max(0,100*xgb_oof_score):.4f}')

In [ ]:
# ─── 9. WEIGHTED ENSEMBLE ─────────────────────────────────────────────────────
# Weight models by their OOF R² score
w_lgb = lgb_oof_score
w_xgb = xgb_oof_score
total_w = w_lgb + w_xgb
w_lgb /= total_w
w_xgb /= total_w

print(f'LGB weight : {w_lgb:.4f}')
print(f'XGB weight : {w_xgb:.4f}')

# OOF blend score
oof_blend  = np.clip(w_lgb * oof_lgb  + w_xgb * oof_xgb,  0, 1)
test_blend = np.clip(w_lgb * test_lgb + w_xgb * test_xgb, 0, 1)

blend_score = r2_score(y, oof_blend)
print(f'\n🏆 Ensemble OOF R²: {blend_score:.6f}  =>  Competition Score: {max(0,100*blend_score):.4f}')

In [ ]:
# ─── 10. FEATURE IMPORTANCE ───────────────────────────────────────────────────
# Retrain final LGB on all data for feature importance plot
final_lgb = lgb.LGBMRegressor(**lgb_params)
final_lgb.set_params(n_estimators=model.best_iteration_ if hasattr(model, 'best_iteration_') else 500)
final_lgb.fit(X, y)

fi = pd.DataFrame({'feature': FEATURES, 'importance': final_lgb.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Feature Importances (LightGBM)', fontsize=13)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(fi.head(10).to_string(index=False))

In [ ]:
# ─── 11. PREDICTION ANALYSIS ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# OOF: Actual vs Predicted
sample_idx = np.random.choice(len(y), 5000, replace=False)
axes[0].scatter(y[sample_idx], oof_blend[sample_idx], alpha=0.3, s=5, color='steelblue')
axes[0].plot([0, 1], [0, 1], 'r--', lw=2)
axes[0].set_xlabel('Actual Demand')
axes[0].set_ylabel('Predicted Demand')
axes[0].set_title(f'OOF: Actual vs Predicted (R²={blend_score:.4f})')

# Test prediction distribution
axes[1].hist(test_blend, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Test Prediction Distribution')
axes[1].set_xlabel('Predicted Demand')

plt.tight_layout()
plt.show()

In [ ]:
# ─── 12. CREATE SUBMISSION FILE ───────────────────────────────────────────────
submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': test_blend
})

submission.to_csv('submission.csv', index=False)

print('Submission saved: submission.csv')
print(f'Shape          : {submission.shape}')
print(f'demand range   : [{submission["demand"].min():.6f}, {submission["demand"].max():.6f}]')
print(f'demand mean    : {submission["demand"].mean():.6f}')
print()
print('Preview:')
print(submission.head(10).to_string(index=False))

In [ ]:
# ─── 13. FINAL SUMMARY ────────────────────────────────────────────────────────
print('=' * 55)
print('           FINAL MODEL SUMMARY')
print('=' * 55)
print(f'  LightGBM OOF R²  : {lgb_oof_score:.6f}  ({max(0,100*lgb_oof_score):.4f})')
print(f'  XGBoost  OOF R²  : {xgb_oof_score:.6f}  ({max(0,100*xgb_oof_score):.4f})')
print(f'  Ensemble OOF R²  : {blend_score:.6f}  ({max(0,100*blend_score):.4f})')
print('=' * 55)
print(f'  Features used    : {len(FEATURES)}')
print(f'  CV folds         : {N_FOLDS}')
print(f'  Submission rows  : {len(submission)}')
print('=' * 55)